# Previsores clássicos: suavização exponencial e SARIMAX

O notebook anterior montou o protocolo: divisão temporal, baselines, horizonte, backtesting e métricas.
Agora entram os dois métodos clássicos que ainda hoje ganham de rede neural em série de negócio:

- **suavização exponencial (ETS)**: o nível, a tendência e a sazonalidade da série, atualizados por média
  ponderada exponencialmente, com peso decrescente para o passado;
- **SARIMAX**: a série explicada pelo próprio passado (parte AR), pelos próprios erros passados (parte MA)
  e por diferenças que removem tendência e sazonalidade, com espaço para variáveis externas.

Os dois são famosos, os dois cabem em poucas linhas de `statsmodels`, e a pergunta deste notebook é a única
que importa: **eles batem o sarrafo do notebook anterior?** O sarrafo é o sazonal ingênuo, e em competição
internacional de previsão (M4) ele costuma ficar entre os primeiros.

A série é a mesma, com os mesmos componentes declarados, para os números dos dois notebooks serem
comparáveis.


In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.api import ExponentialSmoothing, SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller

sns.set_theme(context="notebook", style="whitegrid", palette="colorblind",
              rc={"figure.figsize": (11, 4)})
warnings.filterwarnings("ignore")  # o SARIMAX avisa muito; os avisos são discutidos mais adiante

RANDOM_STATE = 42
PERIODO = 7  # a sazonalidade declarada é semanal


def serie_com_componentes(n_dias=760, seed=RANDOM_STATE):
    """A mesma série do notebook anterior: tendência, semana, ano, ruído e um choque."""
    rng = np.random.default_rng(seed)
    t = np.arange(n_dias)
    tendencia = 100 + 0.06 * t
    sazonalidade_semanal = 6.0 * np.sin(2 * np.pi * t / 7)
    sazonalidade_anual = 12.0 * np.sin(2 * np.pi * t / 365)
    ruido = rng.normal(0, 3.0, n_dias)
    choque = np.where((t >= 400) & (t < 415), 25.0 * np.exp(-(t - 400) / 4.0), 0.0)
    indices = pd.date_range("2024-01-01", periods=n_dias, freq="D")
    return pd.Series(tendencia + sazonalidade_semanal + sazonalidade_anual + ruido + choque,
                     index=indices, name="vendas")


def erro_absoluto_medio(observado, previsto):
    return float(np.mean(np.abs(np.asarray(observado) - np.asarray(previsto))))


def mase(observado, previsto, treino, periodo=PERIODO):
    escala = float(np.mean(np.abs(treino.diff(periodo).dropna())))
    return erro_absoluto_medio(observado, previsto) / escala


def backtest(serie, previsor, origens=9, passos=14):
    """Origem deslizante, com o erro de cada passo guardado para olhar o horizonte depois."""
    por_origem, erro_por_passo = [], []
    for origem in range(len(serie) - passos * origens, len(serie) - passos, passos):
        treino, alvo = serie.iloc[:origem], serie.iloc[origem:origem + passos]
        previsto = previsor(treino, passos)
        por_origem.append({"origem": serie.index[origem].date(),
                           "MAE": erro_absoluto_medio(alvo, previsto),
                           "MASE": mase(alvo, previsto, treino)})
        erro_por_passo.append(np.abs(alvo.to_numpy() - np.asarray(previsto)))
    return pd.DataFrame(por_origem), np.vstack(erro_por_passo)


serie = serie_com_componentes()
treino, teste = serie.iloc[:-14], serie.iloc[-14:]
print(f"série: {len(serie)} dias | treino até {treino.index[-1].date()} | "
      f"teste: {len(teste)} dias")
print(f"o sarrafo do notebook anterior: sazonal ingênuo com MAE 4,332 e MASE 1,215 no treino/teste; "
      f"3,969 de MAE no backtesting")


## Suavização exponencial: três componentes, um previsor

ETS significa Error, Trend, Seasonal: três componentes, cada um podendo entrar de forma **aditiva**
(soma) ou **multiplicativa** (produto). A forma aditiva serve quando a amplitude da sazonalidade é
constante; a multiplicativa, quando a amplitude cresce com o nível da série.

Três escolhas decidem quase tudo:

- **trend**: com ou sem tendência, e a tendência pode ser **amortecida** (damped), crescendo cada vez menos
  conforme o horizonte avança, o que costuma prever melhor;
- **seasonal**: aditiva, multiplicativa ou ausente;
- **períodos**: o tamanho do ciclo, que aqui é 7.

Cabe em uma linha, e é justamente por isso que ele é a linha de base honesta contra métodos caros.


In [ ]:
variantes = {
    "ETS(A,A,A): tendência e semana aditivas": dict(trend="add", seasonal="add",
                                                     seasonal_periods=PERIODO),
    "ETS(A,Ad,A): tendência amortecida": dict(trend="add", damped_trend=True, seasonal="add",
                                              seasonal_periods=PERIODO),
    "ETS(A,A,M): semana multiplicativa": dict(trend="add", seasonal="mul",
                                              seasonal_periods=PERIODO),
    "ETS(A,N,A): sem tendência": dict(trend=None, seasonal="add", seasonal_periods=PERIODO),
}

linhas = []
for nome, config in variantes.items():
    t0 = time.perf_counter()
    modelo = ExponentialSmoothing(treino, **config).fit()
    previsao = modelo.forecast(len(teste)).to_numpy()
    linhas.append({"variante": nome,
                   "MAE": erro_absoluto_medio(teste, previsao),
                   "MASE": mase(teste, previsao, treino),
                   "segundos": time.perf_counter() - t0})

tabela_ets = pd.DataFrame(linhas).set_index("variante").round(3)
display(tabela_ets)

melhor_ets = min(variantes, key=lambda n: tabela_ets.loc[n, "MAE"])
print(f"melhor variante no teste: {melhor_ets}")

sns.lineplot(teste, label="real", color="black")
sns.lineplot(pd.Series(ExponentialSmoothing(treino, **variantes[melhor_ets]).fit()
                       .forecast(len(teste)).to_numpy(), index=teste.index),
             label="ETS escolhido")
plt.title("a melhor suavização exponencial no recorte de teste")
plt.ylabel("vendas")
plt.tight_layout()
plt.show()


Nesta série a suavização exponencial bate o sarrafo com folga: MAE 2,956 e MASE 0,829, contra 4,332 e 1,215
do sazonal ingênuo. E as variantes que parecem mais razoáveis perdem:

- a **tendência amortecida** (3,464) é pior que a linear, porque no horizonte de 14 dias a tendência
  declarada é praticamente reta, e amortecer erra para menos;
- **sem tendência** (4,393) é a pior das quatro, como se espera de uma série que cresce 0,06 por dia;
- a sazonalidade **multiplicativa** (3,017) fica perto da aditiva (2,956), o que é justo: a amplitude
  declarada é constante, então multiplicar não traz vantagem.

Em série cuja amplitude sazonal cresce com o nível, a multiplicativa ganha; aqui, não. É escolha que se
mede. E medir as quatro não custa nada: cada ajuste levou menos de um quarto de segundo.

## Estacionariedade: quando a média e a variância não mudam com o tempo

ARIMA pressupõe que a série é **estacionária**: sem tendência e com o comportamento estatístico estável.
Uma série com tendência não é, e a solução é **diferenciar**: trabalhar com a diferença entre valores
consecutivos em vez dos valores.

O teste ADF (Augmented Dickey-Fuller) dá um p-valor para a hipótese "a série tem raiz unitária", ou seja,
"não é estacionária". P-valor baixo é evidência de estacionariedade.

As funções ACF e PACF mostram a correlação da série com ela mesma em vários atrasos, e são elas que dizem
qual ordem usar: picos no atraso 7, 14, 21 apontam sazonalidade semanal e pedem o termo sazonal.


In [ ]:
for nome, valores in {
    "série original": serie,
    "uma diferença (remove tendência)": serie.diff().dropna(),
    "diferença sazonal de 7 (remove a semana)": serie.diff(PERIODO).dropna(),
    "as duas diferenças": serie.diff(PERIODO).diff().dropna(),
}.items():
    p_valor = adfuller(valores, autolag="AIC")[1]
    print(f"{nome:44s} ADF p-valor {p_valor:.4f} -> "
          f"{'estacionária' if p_valor < 0.05 else 'NÃO estacionária'}")

fig, eixos = plt.subplots(2, 1, figsize=(11, 7))
plot_acf(serie.diff(PERIODO).diff().dropna(), lags=21, ax=eixos[0])
eixos[0].set_title("ACF depois das duas diferenças")
plot_pacf(serie.diff(PERIODO).diff().dropna(), lags=21, ax=eixos[1])
eixos[1].set_title("PACF depois das duas diferenças")
plt.tight_layout()
plt.show()


O teste ADF deu p-valor 0,9588 na série original, ou seja, nenhuma evidência de estacionariedade: a
tendência declarada está lá, e o teste a vê. Uma diferença (0,0000), uma diferença sazonal de 7 (0,0000) e
as duas juntas (0,0000) estacionarizam a série.

A leitura da ACF é a que decide a ordem: os picos aparecem nos múltiplos de sete (7, 14, 21), que é a
assinatura da sazonalidade semanal, e nenhuma diferença simples os remove. É essa leitura que justifica o
parâmetro sazonal `s=7` e a diferença sazonal `D=1` na busca abaixo.

Um cuidado que o teste não resolve: diferenciar demais custa. A diferença remove sinal junto com a
tendência, e é por isso que `d=1` e `D=1` são a escolha aqui, não `d=2`.

## SARIMAX: a busca, e o critério que engana

SARIMAX tem sete parâmetros: `(p, d, q)` para a parte não sazonal e `(P, D, Q, s)` para a sazonal. A busca
abaixo é pequena de propósito, porque cada ajuste custa: são sete combinações, todas com `d=1` e `D=1`.

O `statsmodels` devolve AIC e BIC, que medem o ajuste punindo o número de parâmetros. É tentador escolher
por eles, e é aí que mora a armadilha: AIC e BIC olham **dentro da amostra**, e o que interessa é fora.
A tabela abaixo mostra as duas colunas lado a lado justamente para você ver a discordância.


In [ ]:
combinacoes = [
    ((0, 1, 0), (0, 1, 0, PERIODO)),
    ((1, 1, 0), (0, 1, 0, PERIODO)),
    ((0, 1, 1), (0, 1, 0, PERIODO)),
    ((1, 1, 1), (0, 1, 0, PERIODO)),
    ((1, 1, 1), (1, 1, 0, PERIODO)),
    ((1, 1, 1), (0, 1, 1, PERIODO)),
    ((1, 1, 1), (1, 1, 1, PERIODO)),
]
ordens = {f"{ordem} x {sazonal}": (ordem, sazonal) for ordem, sazonal in combinacoes}

linhas = []
for rotulo, (ordem, ordem_sazonal) in ordens.items():
    t0 = time.perf_counter()
    modelo = SARIMAX(treino, order=ordem, seasonal_order=ordem_sazonal,
                     enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    previsao = modelo.get_forecast(len(teste)).predicted_mean.to_numpy()
    linhas.append({"ordem": rotulo, "AIC": modelo.aic, "BIC": modelo.bic,
                   "MAE": erro_absoluto_medio(teste, previsao),
                   "MASE": mase(teste, previsao, treino),
                   "segundos": time.perf_counter() - t0})

tabela_sarimax = pd.DataFrame(linhas).set_index("ordem").round(3)
display(tabela_sarimax)

melhor_por_aic = tabela_sarimax["AIC"].idxmin()
melhor_por_mae = tabela_sarimax["MAE"].idxmin()
print(f"melhor por AIC: {melhor_por_aic}")
print(f"melhor por MAE no teste: {melhor_por_mae}")


A armadilha está medida na tabela. O melhor AIC é `(1, 1, 1) x (0, 1, 1, 7)`, com 3899,4, e o MAE dele no
teste é 3,763. O melhor MAE é `(1, 1, 1) x (1, 1, 0, 7)`, com 2,887, e o AIC dele é 4112,2, pior que o do
primeiro. O BIC concorda com o AIC, ou seja, trocar de critério não resolve o problema.

Repare no que o termo premiado faz: acrescentar a média móvel sazonal (`Q=1`) melhora o ajuste **dentro** da
amostra em mais de 200 pontos de AIC e piora o erro **fora** em 0,876. É o retrato do excesso de parâmetro:
ele explica o passado, que já foi visto, e não o futuro, que é o que interessa.

Vale ler também a primeira linha: `(0, 1, 0) x (0, 1, 0, 7)` é só as duas diferenças, sem AR nem MA, e
devolve MAE 10,135. Diferenciar remove tendência e sazonalidade, mas não prevê: é quase o mesmo que repetir
o último valor da série diferenciada.

O custo é outro critério a considerar: cada ajuste levou de 0,026 a 0,649 segundo, o mais caro sendo o de
mais parâmetros. Numa série de 760 pontos isso é irrelevante; numa carteira de mil séries, a conta muda.

## O confronto honesto: os dois clássicos contra o sarrafo

O recorte de 14 dias é uma janela só, e o notebook anterior mostrou que a janela muda a resposta. Então a
decisão final usa o mesmo backtesting de nove origens, com o sazonal ingênuo ao lado dos dois clássicos.


In [ ]:
ordem_escolhida = ordens[melhor_por_mae]


def previsor_sarimax(t, h):
    return SARIMAX(t, order=ordem_escolhida[0], seasonal_order=ordem_escolhida[1],
                   enforce_stationarity=False, enforce_invertibility=False
                   ).fit(disp=False).get_forecast(h).predicted_mean.to_numpy()


_, passos_ingenuo = backtest(serie, lambda t, h: np.resize(t.iloc[-PERIODO:].to_numpy(), h))
_, passos_ets = backtest(serie, lambda t, h: ExponentialSmoothing(t, **variantes[melhor_ets])
                         .fit().forecast(h).to_numpy())
_, passos_sarimax = backtest(serie, previsor_sarimax)

comparacao = pd.DataFrame({
    "média do MAE nas origens": [passos_ingenuo.mean(), passos_ets.mean(),
                                 passos_sarimax.mean()],
    "MAE no passo 1": [passos_ingenuo[:, 0].mean(), passos_ets[:, 0].mean(),
                       passos_sarimax[:, 0].mean()],
    "MAE no passo 14": [passos_ingenuo[:, -1].mean(), passos_ets[:, -1].mean(),
                        passos_sarimax[:, -1].mean()],
}, index=["sazonal ingênuo", f"ETS ({melhor_ets.split(':')[0]})", "SARIMAX"]).round(3)
display(comparacao)

comparacao[["MAE no passo 1", "MAE no passo 14"]].plot(kind="bar")
plt.title("erro no primeiro e no último passo de cada horizonte")
plt.ylabel("MAE")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


Nesta série os dois clássicos ganham do sarrafo, e com folga: o ETS faz MAE médio 2,947 e o SARIMAX 3,395,
contra 4,040 do sazonal ingênuo. Isso não é regra geral: no M4, competição com 100 mil séries, o sazonal
ingênuo ficou entre os primeiros e muitas redes neurais ficaram atrás dele. O que faz a diferença aqui é a
série ter exatamente os componentes que os métodos supõem, tendência quase linear e semana estável.

As duas últimas colunas mostram o que a média esconde. No passo 1 o ingênuo é o pior dos três (4,823) e no
passo 14 ele **melhora** (3,791), porque repetir o mesmo dia da semana funciona bem justamente a sete
passos. O SARIMAX faz o caminho oposto: 3,111 no passo 1 e 2,713 no passo 14, porque ele não prevê passo a
passo, ele resolve o horizonte inteiro de uma vez, e por isso não acumula erro. É o oposto exato do boosting
recursivo do notebook anterior. O ETS fica no meio, estável (2,891 e 3,070): ele estende a tendência e
repete a sazonalidade, sem realimentar nada.

A escolha entre os dois, então, não sai da média: o ETS ganha no conjunto e o SARIMAX ganha no passo 14, que
é onde a decisão costuma ser tomada. Medir o horizonte, e não só a média, é o que permite decidir.

## Os resíduos: o que o modelo não explicou

Um modelo de série temporal não termina no erro de previsão. Se sobrou estrutura nos resíduos, ou seja, se
eles ainda têm autocorrelação, então havia informação que o modelo deixou na mesa. O teste de Ljung-Box
mede exatamente isso: p-valor baixo indica resíduo com estrutura, ou seja, modelo mal especificado.


In [ ]:
modelo_final = SARIMAX(treino, order=ordem_escolhida[0], seasonal_order=ordem_escolhida[1],
                       enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

residuos = pd.Series(modelo_final.resid, index=treino.index)
print("Ljung-Box nos resíduos (atrasos 7 e 14):")
print(acorr_ljungbox(residuos, lags=[7, 14], return_df=True).round(4))

sns.lineplot(residuos, color="black")
plt.axhline(0, color="firebrick", linestyle="--")
plt.title("resíduos do SARIMAX escolhido, no treino")
plt.ylabel("resíduo")
plt.tight_layout()
plt.show()


O Ljung-Box devolveu p-valor 0,0000 nos atrasos 7 e 14: os resíduos **têm estrutura**. Traduzindo: o modelo
venceu no teste de previsão e ainda assim deixou informação na mesa.

O motivo está nos componentes declarados. O SARIMAX foi especificado com sazonalidade de 7, e a série tem
também uma sazonalidade anual de amplitude 12 e um choque de 15 dias. Nada disso cabe em `s=7`, e o que
sobra aparece nos resíduos.

É por isso que se olham as duas coisas: o erro de previsão diz se o modelo serve para prever, e o
diagnóstico dos resíduos diz se ele entendeu a série. Um modelo pode passar no primeiro e falhar no segundo,
e foi o que aconteceu aqui.

## Resumo

Suavização exponencial e SARIMAX são as duas famílias clássicas, e as duas custam pouco código. Neste
notebook: as quatro variantes de ETS foram medidas e a melhor (tendência e semana aditivas, MAE 2,956 e MASE
0,829) bateu o sarrafo do sazonal ingênuo (4,332 e 1,215); a tendência amortecida e a variante sem
tendência perderam; o ADF deu p-valor 0,9588 na série original e 0,0000 depois das diferenças; a ACF mostrou
os picos semanais nos múltiplos de sete, que justificam `s=7` e `D=1`; a busca de SARIMAX expôs a armadilha
de escolher por AIC, cujo vencedor (3899,4) perde no teste (MAE 3,763) para uma ordem com AIC pior e MAE
2,887; e no backtesting os dois clássicos bateram o ingênuo (ETS 2,947 e SARIMAX 3,395 contra 4,040), com o
SARIMAX melhorando no passo 14 e o ingênuo também, por motivos diferentes.

Ficou também a lição que separa os dois diagnósticos: o SARIMAX venceu no teste e reprovou no Ljung-Box
(p-valor 0,0000), porque a sazonalidade anual e o choque não cabem em `s=7`.

**Próximo passo:** o notebook de engenharia de atributos desta pasta, onde os mesmos lags viram colunas para
um modelo de boosting competir de igual para igual.

## Referências

```bibtex
@book{hyndman2021forecasting,
  title     = {Forecasting: Principles and Practice},
  author    = {Hyndman, Rob J. and Athanasopoulos, George},
  edition   = {3},
  publisher = {OTexts},
  year      = {2021},
  url       = {https://otexts.com/fpp3/}
}

@book{box2015time,
  title     = {Time Series Analysis: Forecasting and Control},
  author    = {Box, George E. P. and Jenkins, Gwilym M. and Reinsel, Gregory C. and Ljung, Greta M.},
  edition   = {5},
  publisher = {Wiley},
  year      = {2015},
  doi       = {10.1002/9781118619193}
}

@article{makridakis2020m4,
  title   = {The M4 Competition: 100,000 Time Series and 61 Forecasting Methods},
  author  = {Makridakis, Spyros and Spiliotis, Evangelos and Assimakopoulos, Vassilios},
  journal = {International Journal of Forecasting},
  volume  = {36},
  number  = {1},
  pages   = {54--74},
  year    = {2020},
  doi     = {10.1016/j.ijforecast.2019.04.014}
}

@article{ljung1978measure,
  title   = {On a Measure of Lack of Fit in Time Series Models},
  author  = {Ljung, Greta M. and Box, George E. P.},
  journal = {Biometrika},
  volume  = {65},
  number  = {2},
  pages   = {297--303},
  year    = {1978},
  doi     = {10.1093/biomet/65.2.297}
}
```
